In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================
# 1. PARAMETERIZATION (Databricks Widget)
# ==========================================
dbutils.widgets.text("batch_name", "batch_1", "Target Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Processing Silver Products for Batch: {batch_name}")

# ==========================================
# 2. EXTRACT INCOMING DATA & TRANSLATIONS
# ==========================================
df_products_raw = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_products"
).filter(F.col("_batch_id") == batch_name)

df_translation = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_product_category_name_translation"
)

# ==========================================
# 3. TRANSFORMATIONS & TABLE ENRICHMENT
# ==========================================
# Join with category translation table
df_enriched = df_products_raw.join(
    df_translation, on="product_category_name", how="left"
)

# Format category names: Replace '_' with space and apply Title Case (Initcap)
formatted_pt_category = F.initcap(
    F.regexp_replace(F.trim(F.col("product_category_name")), "_", " ")
)
formatted_en_category = F.initcap(
    F.regexp_replace(
        F.trim(F.col("product_category_name_english")), "_", " "
    )
)

# Clean attributes, cast types, handle nulls, and standardize naming
df_cleaned = df_enriched.select(
    F.trim(F.col("product_id")).alias("product_id"),
    F.coalesce(formatted_pt_category, F.lit("Unknown")).alias(
        "product_category_name_pt"
    ),
    F.coalesce(formatted_en_category, F.lit("Unknown")).alias(
        "category_name"
    ),  # Clean English name for BI
    F.coalesce(F.col("product_name_lenght").cast("int"), F.lit(0)).alias(
        "product_name_length"
    ),
    F.coalesce(
        F.col("product_description_lenght").cast("int"), F.lit(0)
    ).alias("product_description_length"),
    F.coalesce(F.col("product_photos_qty").cast("int"), F.lit(0)).alias(
        "photos_qty"
    ),
    F.coalesce(F.col("product_weight_g").cast("double"), F.lit(0.0)).alias(
        "weight_g"
    ),
    F.coalesce(F.col("product_length_cm").cast("double"), F.lit(0.0)).alias(
        "length_cm"
    ),
    F.coalesce(F.col("product_height_cm").cast("double"), F.lit(0.0)).alias(
        "height_cm"
    ),
    F.coalesce(F.col("product_width_cm").cast("double"), F.lit(0.0)).alias(
        "width_cm"
    ),
).dropDuplicates(["product_id"])

# Compute Row Hash using SHA-256 for change detection
df_hashed = df_cleaned.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("product_category_name_pt"), F.lit("")),
            F.coalesce(F.col("category_name"), F.lit("")),
            F.coalesce(F.col("product_name_length").cast("string"), F.lit("")),
            F.coalesce(
                F.col("product_description_length").cast("string"), F.lit("")
            ),
            F.coalesce(F.col("photos_qty").cast("string"), F.lit("")),
            F.coalesce(F.col("weight_g").cast("string"), F.lit("")),
            F.coalesce(F.col("length_cm").cast("string"), F.lit("")),
            F.coalesce(F.col("height_cm").cast("string"), F.lit("")),
            F.coalesce(F.col("width_cm").cast("string"), F.lit("")),
        ),
        256,
    ),
)

# ==========================================
# 4. TARGET TABLE CHECK & INITIALIZATION
# ==========================================
target_table_name = "ecommerce_lakehouse.silver.silver_products"
table_exists = spark.catalog.tableExists(target_table_name)
current_timestamp = F.current_timestamp()

if not table_exists:
  print("Target table does not exist. Initializing silver_products...")

  df_repartitioned = df_hashed.repartition(1)
  window_spec = Window.orderBy("product_id")

  df_transformed = df_repartitioned.withColumn(
      "product_key",
      F.concat(
          F.lit("P"),
          F.lpad(
              (F.row_number().over(window_spec)).cast("string"), 6, "0"
          ),
      ),
  ).select(
      "product_key",
      "product_id",
      "product_category_name_pt",
      "category_name",
      "product_name_length",
      "product_description_length",
      "photos_qty",
      "weight_g",
      "length_cm",
      "height_cm",
      "width_cm",
      "row_hash",
      current_timestamp.alias("_etl_inserted_at"),
      current_timestamp.alias("_etl_updated_at"),
      F.lit(False).alias("_delete_flag"),
  )

  df_transformed.write.format("delta").mode("overwrite").saveAsTable(
      target_table_name
  )
  print("Successfully initialized silver_products dimension!")

else:
  print("Table exists. Executing incremental upsert merge...")

  max_key_row = spark.sql(
      f"SELECT MAX(CAST(SUBSTRING(product_key, 2) AS INT)) as max_key FROM"
      f" {target_table_name}"
  ).collect()[0]
  max_key_val = (
      max_key_row["max_key"] if max_key_row["max_key"] is not None else 0
  )

  df_incoming = df_hashed.withColumn(
      "temp_seq", F.row_number().over(Window.orderBy("product_id"))
  ).withColumn(
      "new_product_key",
      F.concat(
          F.lit("P"),
          F.lpad(
              (F.col("temp_seq") + F.lit(max_key_val)).cast("string"), 6, "0"
          ),
      ),
  )

  deltaTable = DeltaTable.forName(spark, target_table_name)

  deltaTable.alias("target").merge(
      source=df_incoming.alias("source"),
      condition="target.product_id = source.product_id",
  ).whenMatchedUpdate(
      condition="target.row_hash <> source.row_hash",
      set={
          "product_category_name_pt": F.col("source.product_category_name_pt"),
          "category_name": F.col("source.category_name"),
          "product_name_length": F.col("source.product_name_length"),
          "product_description_length": F.col(
              "source.product_description_length"
          ),
          "photos_qty": F.col("source.photos_qty"),
          "weight_g": F.col("source.weight_g"),
          "length_cm": F.col("source.length_cm"),
          "height_cm": F.col("source.height_cm"),
          "width_cm": F.col("source.width_cm"),
          "row_hash": F.col("source.row_hash"),
          "_etl_updated_at": current_timestamp,
      },
  ).whenNotMatchedInsert(
      values={
          "product_key": F.col("new_product_key"),
          "product_id": F.col("source.product_id"),
          "product_category_name_pt": F.col("source.product_category_name_pt"),
          "category_name": F.col("source.category_name"),
          "product_name_length": F.col("source.product_name_length"),
          "product_description_length": F.col(
              "source.product_description_length"
          ),
          "photos_qty": F.col("source.photos_qty"),
          "weight_g": F.col("source.weight_g"),
          "length_cm": F.col("source.length_cm"),
          "height_cm": F.col("source.height_cm"),
          "width_cm": F.col("source.width_cm"),
          "row_hash": F.col("source.row_hash"),
          "_etl_inserted_at": current_timestamp,
          "_etl_updated_at": current_timestamp,
          "_delete_flag": F.lit(False),
      },
  ).execute()

  print("Silver Products processing completed successfully!")

Processing Silver Products for Batch: batch_1
Target table does not exist. Initializing silver_products...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Successfully initialized silver_products dimension!


In [0]:
%sql
select * from ecommerce_lakehouse.silver.silver_products limit 50

product_key,product_id,product_category_name_pt,category_name,product_name_length,product_description_length,photos_qty,weight_g,length_cm,height_cm,width_cm,row_hash,_etl_inserted_at,_etl_updated_at,_delete_flag
P000001,00066f42aeeb9f3007548bb9d3f33c38,Perfumaria,Perfumery,53,596,6,300.0,20.0,16.0,16.0,e0f805d8b7b94706677943876a6c1d1e2122ae9fe64ac9cb132c442ef59f5726,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000002,00088930e925c41fd95ebfe695fd2655,Automotivo,Auto,56,752,4,1225.0,55.0,10.0,26.0,ac20ead91441f15aa9c1f2dd71733a39f62a5e1999b206a4eb3b5053eba48556,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000003,0009406fd7479715e4bef61dd91f2462,Cama Mesa Banho,Bed Bath Table,50,266,2,300.0,45.0,15.0,35.0,37381afe4a53810b78f367763a3d641a9db15d8ca4668bbf127a234059f68c51,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000004,000b8f95fcb9e0096488278317764d19,Utilidades Domesticas,Housewares,25,364,3,550.0,19.0,24.0,12.0,31a71fb0cea9265778f21ef0a7907a87dee3f8aedda4b9c8063a9f89d3044888,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000005,000d9be29b5207b54e86aa1b1ac54872,Relogios Presentes,Watches Gifts,48,613,4,250.0,22.0,11.0,15.0,bd80aae06c905c719c7fbcbe258e7867bc36798698c0149aeb79997a17c8d565,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000006,0011c512eb256aa0dbbb544d8dffcf6e,Automotivo,Auto,58,177,1,100.0,16.0,15.0,16.0,b635bc33335922b4c6fc932b214dbddc6de5256334d6ad0d40a4e8f2ce7804c2,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000007,00126f27c813603687e6ce486d909d01,Cool Stuff,Cool Stuff,42,2461,1,700.0,25.0,5.0,15.0,f8e30073ac1427ace56bb90d390c021d787edd0ad63476f3af62cb2b2a7188ed,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000008,001795ec6f1b187d37335e1c4704762e,Consoles Games,Consoles Games,53,274,1,600.0,30.0,20.0,20.0,cae28ec701659dd965ee2a09d28aee903f254c13573dd8ba1e346d95aa11dfcf,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000009,001b237c0e9bb435f2e54071129237e9,Cama Mesa Banho,Bed Bath Table,42,253,1,6000.0,40.0,4.0,30.0,953ed8caaecf2dfcbc37dcca4cd9b11057fc47b4cc11ed378cab9e2c646fced3,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
P000010,001b72dfd63e9833e8c02742adf472e3,Moveis Decoracao,Furniture Decor,45,520,3,600.0,26.0,8.0,22.0,c517d56f869ad4ccdc921f9b3b539066c2533da39ddd639579645f5ba393b2c1,2026-10-18T06:46:47.933Z,2026-10-18T06:46:47.933Z,false
